In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from collections import Counter

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

using device: cuda


In [2]:
train_dir = r'C:\Users\sagal\Desktop\Let us build\RAF-DB\DATASET\train'

# Seeds
random.seed(42)
torch.manual_seed(42)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.25)) # Increased erasing
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=train_dir, transform=val_transform)

indices = list(range(len(train_dataset)))
random.seed(42)
random.shuffle(indices)

split_size    = int(0.85 * len(indices))
train_indices = indices[:split_size]
val_indices   = indices[split_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset, val_indices)

train_labels = [train_dataset.targets[i] for i in train_indices]
class_counts = Counter(train_labels)
total = len(train_labels)
class_weights = {cls: total / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)

In [4]:
# Load pretrained ResNet-18
resnet = models.resnet18(weights="IMAGENET1K_V1")

# Freeze early layers & layer2; unfreeze layer3, layer4, and fc
for name, param in resnet.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Increased Dropout from 0.5 -> 0.6 to curb FC head memorization
resnet.fc = nn.Sequential(
    nn.Dropout(0.6),
    nn.Linear(resnet.fc.in_features, 7)
)

model = resnet.to(device)

In [5]:
import numpy as np

# --- Mixup Helper Functions ---
def mixup_data(x, y, alpha=0.2):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# --- Setup Training ---
os.makedirs("../../models", exist_ok=True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer configuration
optimizer = Adam([
    {"params": resnet.layer3.parameters(), "lr": 5e-6, "weight_decay": 1e-2},
    {"params": resnet.layer4.parameters(), "lr": 5e-5, "weight_decay": 1e-2},
    {"params": resnet.fc.parameters(),     "lr": 2e-4, "weight_decay": 5e-2}
])

num_epochs = 30
scheduler  = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

patience         = 10
best_val_loss    = float("inf")
patience_counter = 0

save_path = "../../models/best_rafdb_resnet18_enhanced.pth"

# --- Training Loop ---
for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0.0, 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Apply Mixup on input images and labels
        inputs, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.2)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        
        # Calculate weighted training accuracy under Mixup
        correct_a = (outputs.argmax(1) == targets_a).sum().item()
        correct_b = (outputs.argmax(1) == targets_b).sum().item()
        train_correct += (lam * correct_a + (1 - lam) * correct_b)

    # Step learning rate scheduler
    scheduler.step()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    # --- Validation Loop ---
    model.eval()
    val_loss, val_correct = 0.0, 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    current_lr = scheduler.get_last_lr()[-1]
    print(f"Epoch [{epoch+1}/{num_epochs}] (LR: {current_lr:.6f})  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f"  ✓ Best model saved to {save_path} (val loss: {val_loss:.4f}, val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(save_path))
print("Best regularized ResNet-18 model with Mixup loaded!")

Epoch [1/30] (LR: 0.000199)  Train Loss: 1.9125  Train Acc: 28.20%  |  Val Loss: 1.5936  Val Acc: 43.94%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.5936, val acc: 43.94%)
Epoch [2/30] (LR: 0.000198)  Train Loss: 1.6705  Train Acc: 40.60%  |  Val Loss: 1.4725  Val Acc: 49.92%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.4725, val acc: 49.92%)
Epoch [3/30] (LR: 0.000195)  Train Loss: 1.5312  Train Acc: 47.07%  |  Val Loss: 1.2865  Val Acc: 60.24%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.2865, val acc: 60.24%)
Epoch [4/30] (LR: 0.000192)  Train Loss: 1.4668  Train Acc: 51.77%  |  Val Loss: 1.2413  Val Acc: 64.37%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.2413, val acc: 64.37%)
Epoch [5/30] (LR: 0.000187)  Train Loss: 1.4266  Train Acc: 53.61%  |  Val Loss: 1.2230  Val Acc: 64.31%
  ✓ Best model saved to ../../models/best_rafdb_res